In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

### EDA

In [17]:
df = pd.read_csv("data/compustat.csv")
print(df.shape)

(620141, 58)


In [18]:
print(df.duplicated().sum(), "duplicate rows found and deleted")
df = df.drop_duplicates()

0 duplicate rows found and deleted


In [19]:
num_companies = df['gvkey'].dropna().nunique()
print("Number of unique companies:", num_companies)

Number of unique companies: 21048


In [20]:
df['datadate'] = pd.to_datetime(df['datadate'])
df = df.sort_values(['gvkey', 'datadate'])
obs_per_firm = df.groupby('gvkey')['datadate'].count()

# Keep firms with at least 40 quarters (10 years)
valid_firms = obs_per_firm[obs_per_firm >= 12].index
df = df[df['gvkey'].isin(valid_firms)]

In [21]:
targets = ['oancfy', 'cheq']
#Operating Activities – Net Cash Flow (Year-to-Date) = The cumulative cash generated (or used) by operations from the beginning of the fiscal year up to that quarter.
#Cash and Short-Term Investments (Quarterly) = The firm's cash holdings at the end of the quarter.

df[targets].isna().mean()

oancfy    0.337925
cheq      0.316631
dtype: float64

In [22]:
missing = df.isna().mean().sort_values(ascending=False)

missing.head(15)

findltq    0.997650
udoltq     0.988637
scstkcy    0.973903
deraltq    0.890160
derlltq    0.868841
cshopq     0.684350
xrdq       0.680577
xaccq      0.665935
ipodate    0.584330
dd1q       0.529449
npq        0.475765
lltq       0.473970
drcq       0.472951
ivltq      0.460721
wcapq      0.450017
dtype: float64

### preprocessing

In [ ]:

def prepare_features(train_df, test_df, target):
    X_train = train_df.drop(columns=[target, 'datadate', 'gvkey'])
    y_train = train_df[target]
    
    X_test = test_df.drop(columns=[target, 'datadate', 'gvkey'])
    y_test = test_df[target]
    
    # Drop columns with >20% missing
    missing_ratio = X_train.isnull().mean()
    cols_to_keep = missing_ratio[missing_ratio <= 0.20].index
    
    X_train = X_train[cols_to_keep]
    X_test = X_test[cols_to_keep]
    
    # Keep numeric only
    X_train = X_train.select_dtypes(include=['number'])
    X_test = X_test[X_train.columns]
    
    # Impute
    imputer = KNNImputer(n_neighbors=5)
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)
    
    return X_train, X_test, y_train, y_test

In [24]:
macro = pd.read_csv("data/macro.csv")

macro['datadate'] = pd.to_datetime(macro['datadate'])
macro = macro.set_index('datadate').sort_index()

print(macro.head())

                  GDP  IRLTLT01USQ156N    b10ret    t90ret    cpiret    YLDMAT
datadate                                                                      
2014-03-31  17197.738         2.763333  0.036126  0.000222  0.013920  1.543000
2014-06-30  17518.508         2.623333  0.025191  0.000124  0.008676  1.511667
2014-09-30  17804.228         2.496667  0.007790  0.000116 -0.001309  1.583000
2014-12-31  17912.079         2.280000  0.028963  0.000036 -0.013523  1.415333
2015-03-31  18063.529         1.966667  0.024583  0.000091  0.005566  1.324667


In [ ]:
def add_macro_variables(firm_df, macro_df):
    firm_df = firm_df.copy()
    firm_df['datadate'] = pd.to_datetime(firm_df['datadate'])
    
    macro_df.index = pd.to_datetime(macro_df.index)
    
    firm_df = firm_df[
        (firm_df['datadate'] >= "2014-03-31") &
        (firm_df['datadate'] <= "2024-12-31")
    ].copy()
    
    # Merge all macro columns at once
    firm_df = firm_df.merge(
        macro_df,
        left_on="datadate",
        right_index=True,
        how="left"
    )
    
    return firm_df

In [ ]:
def train_random_forest(X_train, y_train):
    model = RandomForestRegressor(
        n_estimators=400,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    return preds, r2, rmse

In [ ]:
def run_model(df, target, use_macro=False, macro_df=None):    
    # Add macro if requested
    if use_macro:
        df = add_macro_variables(df, macro_df)
    
    #drop nan values in target coloumn
    df = df.dropna(subset=[target])

    train_df = df[df['datadate'] <= "2023-12-31"]
    test_df  = df[df['datadate'] > "2023-12-31"]
    
    X_train, X_test, y_train, y_test = prepare_features(
        train_df, test_df, target
    )
    print("training data shape:", X_train.shape)
    
    model = train_random_forest(X_train, y_train)
    preds, r2, rmse = evaluate_model(model, X_test, y_test)
    
    print("Use Macro:", use_macro)
    print("R2:", round(r2,4))
    print("RMSE:", round(rmse,4))
    print("-"*40)
    
    return {
        "model": model,
        "preds": preds,
        "y_test": y_test,
        "r2": r2,
        "rmse": rmse
    }

In [30]:
target = "oancfy"

# WITHOUT macro
results_no_macro = run_model(
    df,
    target,
    use_macro=False
)

KeyboardInterrupt: 

In [ ]:
# WITH macro
results_macro = run_model(
    df,
    target,
    use_macro=True,
    macro_df=macro
)